# Unit 3: Functions & Code Organisation

> **Python Fundamentals**

Complete the reading materials and curated videos before working through this notebook.


## Learning objectives

By the end of this unit you will be able to:

- Identify when a block of code should become a function
- Write functions with parameters, default values, and return values
- Refactor a procedural script into a modular structure


## 1. Why functions?

Imagine you need to calculate a 9% GST amount for several different prices. Without functions, you'd repeat the same calculation over and over:


In [ ]:
price_a = 100
price_b = 250
price_c = 75

gst_a = price_a * 0.09
gst_b = price_b * 0.09
gst_c = price_c * 0.09

print(f"GST on ${price_a}: ${gst_a:.2f}")
print(f"GST on ${price_b}: ${gst_b:.2f}")
print(f"GST on ${price_c}: ${gst_c:.2f}")


This works, but it has a problem: the value `0.09` is repeated three times. If the GST rate changes, you'd need to find and update every occurrence — and it's easy to miss one. This is the **DRY principle**: *Don't Repeat Yourself*.

## 2. Defining a function

A function is a named, reusable block of code. You define it once with `def`, then **call** it as many times as you need.


In [ ]:
def calculate_gst(price):
    gst_rate = 0.09
    return price * gst_rate


gst_a = calculate_gst(100)
gst_b = calculate_gst(250)
gst_c = calculate_gst(75)

print(f"GST on $100: ${gst_a:.2f}")
print(f"GST on $250: ${gst_b:.2f}")
print(f"GST on $75: ${gst_c:.2f}")


Now the GST rate exists in exactly **one place**. If it changes to 10%, you update one line and every call site is correct.

### Anatomy of a function

```python
def calculate_gst(price):   # 'def' + name + parameters in ( )
    gst_rate = 0.09
    return price * gst_rate # 'return' sends a value back to the caller
```

- `price` is a **parameter** — a placeholder for whatever value is passed in
- `return` ends the function and hands a value back to wherever it was called
- Code that calls the function (`calculate_gst(100)`) doesn't need to know *how* the calculation works — only what it needs and what it gets back


## 3. Default parameter values

Sometimes most calls use the same value for a parameter, but you still want to allow it to be overridden occasionally. **Default values** handle this.


In [ ]:
def calculate_gst(price, gst_rate=0.09):
    return price * gst_rate


# Uses the default rate of 9%
print(f"Standard GST: ${calculate_gst(100):.2f}")

# Overrides the rate for a special case
print(f"Special rate GST: ${calculate_gst(100, gst_rate=0.05):.2f}")


## 4. Multiple parameters and return values

Functions can take several parameters, and a `return` statement can send back more than one value at once (Python packages them into a *tuple*).


In [ ]:
def split_bill(total, num_people, gst_rate=0.09):
    total_with_gst = total * (1 + gst_rate)
    per_person = total_with_gst / num_people
    return total_with_gst, per_person


final_total, share = split_bill(120, 4)
print(f"Total with GST: ${final_total:.2f}")
print(f"Each person pays: ${share:.2f}")


## 5. Composing functions

One function can call another. This lets you build complex behaviour out of small, well-tested pieces — each function does one job well.


In [ ]:
def calculate_gst(price, gst_rate=0.09):
    return price * gst_rate


def total_with_gst(price, gst_rate=0.09):
    return price + calculate_gst(price, gst_rate)


def format_currency(amount):
    return f"${amount:,.2f}"


price = 1500
print(f"Price: {format_currency(price)}")
print(f"GST: {format_currency(calculate_gst(price))}")
print(f"Total: {format_currency(total_with_gst(price))}")


## 6. When should a block of code become a function?

Ask yourself:

- **Will I need this logic more than once?** — if yes, it's a strong candidate.
- **Does this block do one clear, nameable thing?** — if you can give it a good name (e.g. `calculate_gst`, `format_currency`), it's probably a function.
- **Is this block more than a few lines and mixed in with unrelated logic?** — pulling it out makes the surrounding code easier to read.
- **Would I want to test this piece of logic on its own?** — functions are easy to test individually; long scripts are not.

## 7. Refactoring example: before and after

**Before** — a procedural script that mixes calculation, formatting, and printing for each item:


In [ ]:
items = [
    {"name": "Laptop", "price": 1500},
    {"name": "Mouse", "price": 25},
    {"name": "Monitor", "price": 300},
]

for item in items:
    name = item["name"]
    price = item["price"]
    gst = price * 0.09
    total = price + gst
    print(f"{name}: price=${price:.2f}, gst=${gst:.2f}, total=${total:.2f}")


**After** — the calculation logic is pulled into functions. The loop now reads almost like plain English, and `calculate_gst`/`total_with_gst` can be reused anywhere else in the program (or tested on their own).


In [ ]:
def calculate_gst(price, gst_rate=0.09):
    return price * gst_rate


def total_with_gst(price, gst_rate=0.09):
    return price + calculate_gst(price, gst_rate)


items = [
    {"name": "Laptop", "price": 1500},
    {"name": "Mouse", "price": 25},
    {"name": "Monitor", "price": 300},
]

for item in items:
    name = item["name"]
    price = item["price"]
    gst = calculate_gst(price)
    total = total_with_gst(price)
    print(f"{name}: price=${price:.2f}, gst=${gst:.2f}, total=${total:.2f}")


## 8. A note on modules and imports

As your collection of useful functions grows, you can move them into a separate `.py` file (a **module**) and `import` them wherever you need them — instead of redefining the same functions in every notebook or script.

```python
# pricing.py
def calculate_gst(price, gst_rate=0.09):
    return price * gst_rate
```

```python
# main.py
from pricing import calculate_gst

print(calculate_gst(100))
```

You won't need this for the exercises in this unit, but it's the natural next step once functions start being reused across multiple files — and you'll see it throughout real-world Python projects.


## 9. Putting it together

The procedural code below calculates a **late fee** for a list of invoices: any invoice more than 30 days overdue gets a 5% late fee added to its total.

Refactor it by writing two functions:

1. `calculate_late_fee(amount, days_overdue, fee_rate=0.05)` — returns the late fee amount, or `0` if `days_overdue <= 30`
2. `invoice_total(amount, days_overdue, fee_rate=0.05)` — returns `amount` plus the late fee

Then rewrite the loop to use your functions.


In [ ]:
invoices = [
    {"id": "INV-001", "amount": 500, "days_overdue": 10},
    {"id": "INV-002", "amount": 1200, "days_overdue": 45},
    {"id": "INV-003", "amount": 80, "days_overdue": 60},
]

# TODO: define calculate_late_fee(amount, days_overdue, fee_rate=0.05)


# TODO: define invoice_total(amount, days_overdue, fee_rate=0.05)


# TODO: loop over invoices and print id, late fee, and total for each
for invoice in invoices:
    pass


## Next steps

Open **`exercises.ipynb`** in this repository to practise what you've learned in this unit.
